# Automated Casting Defect Detection Using CNN

**Binary Image Classification for Quality Control**

This notebook trains a Convolutional Neural Network to detect casting defects in manufactured products.
- Input: Product image (224×224 RGB)
- Output: Defective (1) or Non-defective (0)
- Dataset: 6,633 casting images from Kaggle

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import os

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow: {tf.__version__}")

In [ ]:
# Configuration
image_size = (224, 224)
batch_size = 32
epochs = 25

train_directory = "../../data/train"
test_directory = "../../data/test"
class_names = ["ok_front", "def_front"]

print(f"Train path: {os.path.exists(train_directory)}")
print(f"Test path: {os.path.exists(test_directory)}")

In [ ]:
# Load datasets
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory, class_names=class_names, validation_split=0.20,
    subset="training", seed=42, image_size=image_size, batch_size=batch_size, label_mode="binary"
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory, class_names=class_names, validation_split=0.20,
    subset="validation", seed=42, image_size=image_size, batch_size=batch_size, label_mode="binary"
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_directory, class_names=class_names, image_size=image_size,
    batch_size=batch_size, label_mode="binary", shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(AUTOTUNE)
validation_dataset = validation_dataset.prefetch(AUTOTUNE)
test_dataset = test_dataset.prefetch(AUTOTUNE)

In [ ]:
# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),
    layers.RandomContrast(0.10)
], name="data_augmentation")

In [ ]:
# Build CNN
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    data_augmentation,
    layers.Rescaling(1.0 / 255),
    
    layers.Conv2D(32, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.40),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(1, activation="sigmoid")
], name="casting_defect_detector")

model.summary()

In [ ]:
# Compile
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"), 
             tf.keras.metrics.Recall(name="recall")]
)

In [ ]:
# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=0.000001),
    tf.keras.callbacks.ModelCheckpoint(filepath="../../models/best_casting_defect_model.keras", 
                                       monitor="val_loss", save_best_only=True)
]

In [ ]:
# Train
history = model.fit(train_dataset, validation_data=validation_dataset, 
                   epochs=epochs, callbacks=callbacks, verbose=1)

In [ ]:
# Plot training results
epochs_range = range(1, len(history.history["accuracy"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_range, history.history["accuracy"], label="Train")
axes[0].plot(epochs_range, history.history["val_accuracy"], label="Val")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(epochs_range, history.history["loss"], label="Train")
axes[1].plot(epochs_range, history.history["val_loss"], label="Val")
axes[1].set_title("Loss")
axes[1].legend()
plt.tight_layout()
plt.savefig("../../reports/training_results.png", dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluate
test_results = model.evaluate(test_dataset, verbose=0)
print(f"Test Accuracy: {test_results[1]:.4f}")

# Predictions
prediction_probabilities = model.predict(test_dataset, verbose=0)
predicted_labels = (prediction_probabilities.flatten() >= 0.5).astype(int)
actual_labels = np.concatenate([labels.numpy().flatten() for images, labels in test_dataset]).astype(int)

print(classification_report(actual_labels, predicted_labels, target_names=["Non-defective", "Defective"]))

# Confusion matrix
cm = confusion_matrix(actual_labels, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("../../reports/confusion_matrix.png", dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Save model
model.save("../../models/casting_defect_model.keras")
print("✅ Model saved to: models/casting_defect_model.keras")

In [ ]:
def predict_product(image_path, model, threshold=0.50):
    """Predict if a product image shows defects"""
    image = tf.keras.utils.load_img(image_path, target_size=(224, 224))
    image_array = tf.keras.utils.img_to_array(image)
    image_array = tf.expand_dims(image_array, axis=0)
    
    defect_prob = float(model.predict(image_array, verbose=0)[0][0])
    
    prediction = "Defective" if defect_prob >= threshold else "Non-defective"
    action = "Send for manual inspection" if defect_prob >= threshold else "Product may proceed"
    
    print(f"Prediction: {prediction}")
    print(f"Defect probability: {defect_prob:.2%}")
    print(f"Action: {action}")
    
    return prediction

# Example: predict_product("sample_images/product.jpeg", model, threshold=0.50)

## Summary

**Model Performance:**
- Trained on 6,633 casting images
- Test accuracy, precision, recall shown above
- Confusion matrix saved to reports/

**Files Generated:**
- `models/casting_defect_model.keras` - Trained model
- `models/best_casting_defect_model.keras` - Best weights
- `reports/training_results.png` - Accuracy/Loss graphs
- `reports/confusion_matrix.png` - Confusion matrix

**Next Steps:**
1. Load model: `model = tf.keras.models.load_model('models/casting_defect_model.keras')`
2. Predict: Use `predict_product()` function
3. Deploy: Integrate with production pipeline